# fp growth

FP-Growth is an unsupervised association-rule-learning algorithm that finds frequent itemsets in transaction data without generating candidate itemsets like Apriori does.


# For more info click the link below

https://docs.google.com/document/d/1z1d2-Vp2dnuMsMs-TPnwPKQDkv28OfAkOpAbT8URQRA/edit?usp=sharing

# Notes and interview Q and anawer

https://docs.google.com/document/d/1jwpsxZFBBdX03cOQ8Wv5KowNhTRKTPit3QVwGR0-Z1M/edit?usp=sharing

# Coding

In [16]:
import pandas as pd                                      # Data manipulation

from mlxtend.preprocessing import TransactionEncoder    # Convert transactions into one-hot format
from mlxtend.frequent_patterns import fpgrowth          # FP-Growth frequent-itemset algorithm
from mlxtend.frequent_patterns import association_rules # Generate association rules

# Create transaction data 

In [17]:
transactions = [
    ["Milk", "Bread", "Butter"],                         # Transaction 1
    ["Milk", "Bread"],                                   # Transaction 2
    ["Milk", "Bread", "Butter", "Eggs"],                 # Transaction 3
    ["Bread", "Butter"],                                 # Transaction 4
    ["Milk", "Bread", "Eggs"],                           # Transaction 5
    ["Milk", "Butter"],                                  # Transaction 6
    ["Bread", "Butter", "Eggs"],                         # Transaction 7
    ["Milk", "Bread", "Butter"],                         # Transaction 8
    ["Milk", "Bread", "Eggs"],                           # Transaction 9
    ["Bread", "Butter"]                                  # Transaction 10
]

# Convert transaction into one-hot encoding

In [18]:
encoder = TransactionEncoder()                           # Create transaction-to-binary encoder

encoded_array = encoder.fit(
    transactions                                            # Learn all unique items
).transform(
    transactions                                            # Convert transactions into True/False values
)

df = pd.DataFrame(
    encoded_array,                                         # Encoded transaction matrix
    columns=encoder.columns_                               # Use item names as column names
)

print(df)

   Bread  Butter   Eggs   Milk
0   True    True  False   True
1   True   False  False   True
2   True    True   True   True
3   True    True  False  False
4   True   False   True   True
5  False    True  False   True
6   True    True   True  False
7   True    True  False   True
8   True   False   True   True
9   True    True  False  False


# Run FP-Growth 
`Now the actual algorithm:`

In [19]:
frequent_itemsets = fpgrowth(
    df,                                                    # Transaction data in one-hot format
    min_support=0.3,                                      # Itemset must appear in at least 30% of transactions
    use_colnames=True                                     # Return actual item names instead of column numbers
)

print(frequent_itemsets)

    support               itemsets
0       0.9                (Bread)
1       0.7                 (Milk)
2       0.7               (Butter)
3       0.4                 (Eggs)
4       0.6          (Milk, Bread)
5       0.6        (Butter, Bread)
6       0.4         (Butter, Milk)
7       0.3  (Butter, Milk, Bread)
8       0.4          (Eggs, Bread)
9       0.3           (Milk, Eggs)
10      0.3    (Milk, Eggs, Bread)


# Sort frequent intemsets 

In [20]:
frequent_itemsets = frequent_itemsets.sort_values(
    by="support",                                         # Sort according to frequency
    ascending=False                                       # Highest support first
)

print(frequent_itemsets)

    support               itemsets
0       0.9                (Bread)
1       0.7                 (Milk)
2       0.7               (Butter)
4       0.6          (Milk, Bread)
5       0.6        (Butter, Bread)
3       0.4                 (Eggs)
6       0.4         (Butter, Milk)
8       0.4          (Eggs, Bread)
7       0.3  (Butter, Milk, Bread)
9       0.3           (Milk, Eggs)
10      0.3    (Milk, Eggs, Bread)


# Generate association rules 

In [21]:
rules = association_rules(
    frequent_itemsets,                                    # Frequent itemsets discovered by FP-Growth
    metric="confidence",                                  # Use confidence to filter rules
    min_threshold=0.6                                    # Keep rules with at least 60% confidence
)

print(rules)

      antecedents    consequents  antecedent support  consequent support  \
0          (Milk)        (Bread)                 0.7                 0.9   
1         (Bread)         (Milk)                 0.9                 0.7   
2        (Butter)        (Bread)                 0.7                 0.9   
3         (Bread)       (Butter)                 0.9                 0.7   
4          (Eggs)        (Bread)                 0.4                 0.9   
5  (Butter, Milk)        (Bread)                 0.4                 0.9   
6          (Eggs)         (Milk)                 0.4                 0.7   
7    (Milk, Eggs)        (Bread)                 0.3                 0.9   
8   (Eggs, Bread)         (Milk)                 0.4                 0.7   
9          (Eggs)  (Milk, Bread)                 0.4                 0.6   

   support  confidence      lift  representativity  leverage  conviction  \
0      0.6    0.857143  0.952381               1.0     -0.03         0.7   
1      0.6 

# Show the most interesting rules 

In [22]:
rules = rules.sort_values(
    by="lift",                                             # Sort according to association strength
    ascending=False                                       # Strongest rules first
)

print(
    rules[
        [
            "antecedents",                                 # Items on the left side of the rule
            "consequents",                                 # Items on the right side of the rule
            "support",                                     # Frequency of the complete itemset
            "confidence",                                  # Probability of consequent given antecedent
            "lift"                                         # Strength compared with independent occurrence
        ]
    ]
)

      antecedents    consequents  support  confidence      lift
9          (Eggs)  (Milk, Bread)      0.3    0.750000  1.250000
4          (Eggs)        (Bread)      0.4    1.000000  1.111111
7    (Milk, Eggs)        (Bread)      0.3    1.000000  1.111111
6          (Eggs)         (Milk)      0.3    0.750000  1.071429
8   (Eggs, Bread)         (Milk)      0.3    0.750000  1.071429
0          (Milk)        (Bread)      0.6    0.857143  0.952381
1         (Bread)         (Milk)      0.6    0.666667  0.952381
2        (Butter)        (Bread)      0.6    0.857143  0.952381
3         (Bread)       (Butter)      0.6    0.666667  0.952381
5  (Butter, Milk)        (Bread)      0.3    0.750000  0.833333


# Experiment with min_support 

`THis is the main FP-Growth eexperiment`

In [23]:
for support in [0.2, 0.3, 0.4, 0.5]:

    frequent_itemsets = fpgrowth(
        df,                                                # Transaction dataset
        min_support=support,                               # Change minimum frequency requirement
        use_colnames=True                                  # Return actual item names
    )

    print(
        f"min_support={support} → "
        f"itemsets={len(frequent_itemsets)}"
    )

min_support=0.2 → itemsets=13
min_support=0.3 → itemsets=11
min_support=0.4 → itemsets=8
min_support=0.5 → itemsets=5


# Complete end-to-end

In [24]:
import pandas as pd                                      # Data manipulation

from mlxtend.preprocessing import TransactionEncoder    # Convert transactions into one-hot format
from mlxtend.frequent_patterns import fpgrowth          # FP-Growth frequent-itemset algorithm
from mlxtend.frequent_patterns import association_rules # Generate association rules


transactions = [
    ["Milk", "Bread", "Butter"],                         # Transaction 1
    ["Milk", "Bread"],                                   # Transaction 2
    ["Milk", "Bread", "Butter", "Eggs"],                 # Transaction 3
    ["Bread", "Butter"],                                 # Transaction 4
    ["Milk", "Bread", "Eggs"],                           # Transaction 5
    ["Milk", "Butter"],                                  # Transaction 6
    ["Bread", "Butter", "Eggs"],                         # Transaction 7
    ["Milk", "Bread", "Butter"],                         # Transaction 8
    ["Milk", "Bread", "Eggs"],                           # Transaction 9
    ["Bread", "Butter"]                                  # Transaction 10
]


encoder = TransactionEncoder()                           # Create transaction encoder

encoded_array = encoder.fit(
    transactions                                             # Learn unique products
).transform(
    transactions                                             # Convert transactions to binary format
)


df = pd.DataFrame(
    encoded_array,                                         # Encoded transaction matrix
    columns=encoder.columns_                               # Product names as columns
)


frequent_itemsets = fpgrowth(
    df,                                                    # Transaction dataset
    min_support=0.3,                                      # Keep itemsets appearing in at least 30% of transactions
    use_colnames=True                                     # Return product names
)


print("\nFrequent Itemsets:")
print(frequent_itemsets)


rules = association_rules(
    frequent_itemsets,                                    # Frequent itemsets discovered by FP-Growth
    metric="confidence",                                  # Use confidence to filter rules
    min_threshold=0.6                                    # Keep rules with at least 60% confidence
)


rules = rules.sort_values(
    by="lift",                                            # Sort rules by association strength
    ascending=False                                       # Strongest rules first
)


print("\nAssociation Rules:")
print(
    rules[
        [
            "antecedents",                                # Items on the left side
            "consequents",                                # Items on the right side
            "support",                                    # Frequency of complete itemset
            "confidence",                                 # Probability of consequent given antecedent
            "lift"                                        # Strength of association
        ]
    ]
)


Frequent Itemsets:
    support               itemsets
0       0.9                (Bread)
1       0.7                 (Milk)
2       0.7               (Butter)
3       0.4                 (Eggs)
4       0.6          (Milk, Bread)
5       0.6        (Butter, Bread)
6       0.4         (Butter, Milk)
7       0.3  (Butter, Milk, Bread)
8       0.4          (Eggs, Bread)
9       0.3           (Milk, Eggs)
10      0.3    (Milk, Eggs, Bread)

Association Rules:
      antecedents    consequents  support  confidence      lift
9          (Eggs)  (Milk, Bread)      0.3    0.750000  1.250000
5          (Eggs)        (Bread)      0.4    1.000000  1.111111
7    (Milk, Eggs)        (Bread)      0.3    1.000000  1.111111
6          (Eggs)         (Milk)      0.3    0.750000  1.071429
8   (Eggs, Bread)         (Milk)      0.3    0.750000  1.071429
0          (Milk)        (Bread)      0.6    0.857143  0.952381
1         (Bread)         (Milk)      0.6    0.666667  0.952381
2        (Butter)        (Br